In [2]:
def large_discrete(n_samples=100_000):

    X1 = np.random.binomial(1, 0.5, n_samples)  + np.random.binomial(1, 0.1, n_samples) # Add noise

    X2 = np.random.binomial(1, 0.5, n_samples) + np.random.binomial(1, 0.1, n_samples)  # Add noise

    X3 = X1 ^ X2 + np.random.binomial(1, 0.1, n_samples)  # Add noise

    X4 = np.random.binomial(1, 0.1, n_samples)

    X5 = X4 + X3 + np.random.binomial(1, 0.1, n_samples) # Add noise

    X6 = X4*X3 + np.random.binomial(1, 0.1, n_samples)  # Add noise

    X7 = X1 + X4 + np.random.binomial(1, 0.1, n_samples)  # Add noise

    X8 = np.random.binomial(1, 0.5, n_samples) 

    X9 =  np.random.binomial(1, 0.5, n_samples)

    return pd.DataFrame({'X1': X1, 'X2': X2, 'X3': X3, 'X4': X4, 'X5': X5, 'X6': X6, 'X7': X7, 'X8': X8, 'X9': X9})

In [3]:
import numpy as np
import pandas as pd

large_discrete_data = large_discrete(10000)
model =DiscreteBayesianNetwork([
    ("X1", "X3"),
    ("X2", "X3"),
    ("X4", "X5"),
    ("X3", "X5"),
    ("X4", "X6"),
    ("X3", "X6"),
    ("X1", "X7"),
    ("X4", "X7"),
])

# triplets=[("X1","X2","X3"),("X3","X4","X5"),("X3","X4","X6"),("X1","X4","X7")]

model.add_node("X8")
model.add_node("X9")

score = BIC(large_discrete_data)

hc = HillClimbSearch(large_discrete_data)
# hcc = HillClimbCustom(large_discrete_data)
dj = DoubleJumpHillClimbSearch(large_discrete_data)
wms = DoubleJumpHillClimbSearchWMS(large_discrete_data)
ii = dag_ii(large_discrete_data)


large = hc.estimate(scoring_method=score)
# large_custom = hcc.estimate(scoring_method=score)
dj_large = dj.estimate(scoring_method=score)
# wms_large = wms.estimate(scoring_method=score,
#                          data=large_discrete_data,
#                          preprocess=True,
# )


visualize_model(model, "truth")
visualize_model(large, "hc")
visualize_model(large_custom, "hcc")
visualize_model(dj_large, "dj")
# visualize_model(wms_large, "wms")
visualize_model(ii, "ii")

INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'X1': 'N', 'X2': 'N', 'X3': 'N', 'X4': 'N', 'X5': 'N', 'X6': 'N', 'X7': 'N', 'X8': 'N', 'X9': 'N'}
INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'X1': 'N', 'X2': 'N', 'X3': 'N', 'X4': 'N', 'X5': 'N', 'X6': 'N', 'X7': 'N', 'X8': 'N', 'X9': 'N'}


NameError: name 'DoubleJumpHillClimbSearch' is not defined

In [ ]:
from src.utils import compare_dag

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_selection import mutual_info_classif

# Generate example DataFrame with 9 variables
# data = fork_discrete_d(10000)

# Function to compute all pairwise mutual information
def compute_pairwise_mi(df):
    columns = df.columns
    n = len(columns)
    mi_matrix = pd.DataFrame(np.zeros((n, n)), index=columns, columns=columns)
    
    for i, col1 in enumerate(columns):
        for j, col2 in enumerate(columns):
            if i < j:  # Compute MI only for upper triangular part
                mi = mutual_info_classif(df[[col1]], df[col2])[0]
                mi_matrix.loc[col1, col2] = mi
                mi_matrix.loc[col2, col1] = mi  # Symmetric matrix
    
    return mi_matrix

# Function to calculate the whole mutual information I(A, B, C)
def whole_mutual_information(df, col1, col2, col3):
    combined_abc = df[[col1, col2]].apply(tuple, axis=1).astype('category').cat.codes
    whole_mi = mutual_info_classif(pd.DataFrame(combined_abc), df[col3])[0]
    return whole_mi

def find_triplets(data):
    # Compute pairwise mutual information
    mi_matrix = compute_pairwise_mi(data)

    # Find synergistic triplets
    synergistic_triplets = []
    columns = data.columns

    for i, col1 in enumerate(columns):
        for j, col2 in enumerate(columns):
            for k, col3 in enumerate(columns):
                if len(set([col1, col2, col3])) == 3:  # Ensure unique triplet
                    whole_mi = whole_mutual_information(data, col1, col2, col3)
                    pairwise_mi_sum = (
                        mi_matrix.loc[col1, col2]
                        + mi_matrix.loc[col2, col3]
                        + mi_matrix.loc[col1, col3]
                    )
                    synergy = whole_mi - pairwise_mi_sum
                    if synergy > 0.09:  # Check if synergistic
                        synergistic_triplets.append([col1, col2, col3])

    # Use frozenset to ensure uniqueness (order doesn't matter)
    unique_triplets = list({frozenset(triplet) for triplet in synergistic_triplets})
    # Convert back to list format (if needed)
    unique_triplets = [list(triplet) for triplet in unique_triplets]
    # Convert results to DataFrame for better visualization
    results_df = pd.DataFrame(unique_triplets)
#     print(unique_triplets)

    # Display results
#     if not results_df.empty:
#         print("Synergistic Triplets Found:")
#         print(results_df)
#     else:
#         print("No synergistic triplets found.")
    return unique_triplets

# find_triplets(collider_discrete_d(10000))